In [1]:
import pandas as pd
import numpy as np

In [2]:
# Load the data

df = pd.read_csv("powerplant_data.csv")

In [3]:

X = df.drop("PE", axis=1)
y = df["PE"]

In [4]:
# split the data

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [5]:
df.shape

(9568, 5)

In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [7]:
X_test_scaled

array([[ 1.34499288,  0.23869298, -1.28658067, -1.10532538],
       [ 0.81095912,  1.36269098, -0.74140656,  0.26485915],
       [-0.2437241 , -0.73900436,  1.99970178, -0.19713193],
       ...,
       [-0.67068342, -1.15902881, -0.29951077, -0.10651852],
       [ 1.31420898,  1.33752097, -0.87346737, -0.44288647],
       [-0.2611237 , -0.27021304,  0.37433797,  1.10646548]],
      shape=(1914, 4))

In [8]:
import torch
import torch.nn as nn

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1,1)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1,1)

In [9]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [10]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

Deep Learning 

In [11]:
class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()

        self.model = nn.Sequential(
            # 1st hidden layer
            nn.Linear(X_train.shape[1], 6),
            nn.ReLU(),

            # 2nd hidden layer
            nn.Linear(6,6),
            nn.ReLU(),

            # output layer
            nn.Linear(6,1),
        )

    def forward(self, x):
        return self.model(x)


        

In [12]:
import torch.optim as optim

model = ANN()

# loss and optimizer

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters())

In [27]:
# Train the ANN
train_losses = []
val_losses = []

best_val_loss = float("inf")

epochs = 100

for epoch in range(epochs):
    model.train()
    running_loss = 0.0 # tot training loss for 1 epoch
    
    for xb, yb in train_loader:
        # xb = features of 1 batch
        # yb = labels of 1 batch
        optimizer.zero_grad()
        
        outputs = model(xb) # forward prop....predicted outputs for this batch
        loss = criterion(outputs, yb) # compute loss
        loss.backward() # back prop.. compute gradients
        optimizer.step() # params update
        
        running_loss += loss.item() # loss is a tensor => py float

    epoch_train_loss = running_loss / len(train_loader)
    train_losses.append(epoch_train_loss)


    # Validation
    model.eval()
    running_val_loss = 0.0

    with torch.no_grad(): # no gradients compute
        for xb, yb in test_loader:
            outputs = model(xb)
            loss = criterion(outputs, yb)
            running_val_loss += loss

    epoch_val_loss = running_val_loss / len(test_loader)
    val_losses.append(epoch_val_loss)

    print(f"epoch {epoch+1}/{epochs} ==> train loss = {epoch_train_loss} & val loss = {epoch_val_loss}")

    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), "best_model.pt") #.pt or .pth

epoch 1/100 ==> train loss = 188814.33196614584 & val loss = 169772.25
epoch 2/100 ==> train loss = 141761.9501953125 & val loss = 112348.0859375
epoch 3/100 ==> train loss = 87792.30616861979 & val loss = 68349.109375
epoch 4/100 ==> train loss = 54580.659655761716 & val loss = 43108.1640625
epoch 5/100 ==> train loss = 32583.50388183594 & val loss = 23177.62109375
epoch 6/100 ==> train loss = 16038.741394042969 & val loss = 10420.7412109375
epoch 7/100 ==> train loss = 7422.132480875651 & val loss = 5458.62255859375
epoch 8/100 ==> train loss = 4539.361113484701 & val loss = 3924.393310546875
epoch 9/100 ==> train loss = 3419.1345006306965 & val loss = 3054.549072265625
epoch 10/100 ==> train loss = 2653.442707824707 & val loss = 2401.1220703125
epoch 11/100 ==> train loss = 2085.5582270304362 & val loss = 1899.723388671875
epoch 12/100 ==> train loss = 1656.148860168457 & val loss = 1523.203125
epoch 13/100 ==> train loss = 1336.4852058410645 & val loss = 1246.435302734375
epoch 14/